In [1]:
!pip install -q -U "protobuf<4"
!pip install -q -U bitsandbytes peft accelerate datasets transformers sacrebleu evaluate

In [ ]:
import os
import sys

# --- 1. THIẾT LẬP MÔI TRƯỜNG ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"

import torch

if torch.cuda.device_count() > 1:
    raise RuntimeError("LỖI: Máy vẫn đang nhận diện 2 GPU! Factory reset và chạy lại.")
print(f"✅ Hệ thống: {torch.cuda.device_count()} GPU")

# --- 2. IMPORT ---
from datasets import Dataset
import re
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --- 3. CONFIG ---
class config:
    data_dir = "/kaggle/input/vlsp-data" 
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    output_dir = "/kaggle/working/qwen_mt_vi_en_v2"
    
    
    checkpoint_dataset_path = "/kaggle/input/checkpoint/"  
    
   
    lora_r = 32
    lora_alpha = 64
    lora_dropout = 0.05
    
    
    num_epochs = 3          
    batch_size = 8           
    gradient_accumulation_steps = 4  
    learning_rate = 2e-4     
    
   
    max_train_samples = 150000  
    
# --- 4. XỬ LÝ DỮ LIỆU ---
def load_data(src_filename, trg_filename):
    src_path, trg_path = "", ""
    for root, dirs, files in os.walk(config.data_dir):
        if src_filename in files: src_path = os.path.join(root, src_filename)
        if trg_filename in files: trg_path = os.path.join(root, trg_filename)
            
    if not src_path or not trg_path: 
        raise FileNotFoundError("Không tìm thấy file!")

    with open(src_path, 'r', encoding='utf-8') as f: 
        src_lines = [l.strip() for l in f]
    with open(trg_path, 'r', encoding='utf-8') as f: 
        trg_lines = [l.strip() for l in f]
    return list(zip(src_lines, trg_lines))

def clean_text(text):
    text = ' '.join(text.split())
    return re.sub(r'\s+([.,!?;:])', r'\1', text).strip()

def create_instruction_format(src, trg):
    
    sys = "Translate Vietnamese medical text to English accurately and concisely."
    user = src  
    
    return {
        "messages": [
            {"role": "system", "content": sys}, 
            {"role": "user", "content": user}, 
            {"role": "assistant", "content": trg}
        ]
    }

# --- 5. CHUẨN BỊ DATASET ---
print("📂 Loading dữ liệu VI-EN...")
data_raw = load_data("train.vi.txt", "train.en.txt")


if config.max_train_samples:
    data_raw = data_raw[:config.max_train_samples]
    print(f"📊 Sử dụng {len(data_raw)} samples")
else:
    print(f"📊 Sử dụng TOÀN BỘ {len(data_raw)} samples")

train_raw, val_raw = train_test_split(data_raw, test_size=0.05, random_state=42)  # 0.1 → 0.05

train_dataset = Dataset.from_list([
    create_instruction_format(clean_text(s), clean_text(t)) 
    for s, t in train_raw if s and t
])

val_dataset = Dataset.from_list([
    create_instruction_format(clean_text(s), clean_text(t)) 
    for s, t in val_raw if s and t
])

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# Verify
print("\n=== VERIFY PROMPT ===")
print(f"System: {train_dataset[0]['messages'][0]['content']}")
print(f"User: {train_dataset[0]['messages'][1]['content'][:100]}...")
print(f"Assistant: {train_dataset[0]['messages'][2]['content'][:100]}...")
print("=" * 50 + "\n")

# --- 6. LOAD TOKENIZER & MODEL ---
tokenizer = AutoTokenizer.from_pretrained(
    config.model_name, 
    trust_remote_code=True, 
    padding_side="left" 
)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(examples):
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) 
        for m in examples["messages"]
    ]
    tokenized = tokenizer(
        texts, 
        truncation=True, 
        max_length=256, 
        padding="max_length", 
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    tokenized["labels"][tokenized["labels"] == tokenizer.pad_token_id] = -100
    return tokenized

print("🔄 Tokenizing datasets...")
train_dataset = train_dataset.map(format_chat_template, batched=True)
val_dataset = val_dataset.map(format_chat_template, batched=True)

print("🔄 Loading Model 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=False
)

model = AutoModelForCausalLM.from_pretrained(
    config.model_name, 
    quantization_config=bnb_config, 
    device_map="auto", 
    trust_remote_code=True
)

model.config.use_cache = False 
model = prepare_model_for_kbit_training(model)


peft_config = LoraConfig(
    r=config.lora_r,                
    lora_alpha=config.lora_alpha,   
    lora_dropout=config.lora_dropout,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none", 
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# --- 7. TRAINING ARGS--
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    fp16=True,
    
    
    warmup_steps=200,        
    lr_scheduler_type="cosine",
    
    
    logging_steps=500,       
    logging_dir=os.path.join(config.output_dir, "logs"),
    
    
    eval_strategy="steps",
    eval_steps=2000,         

    
    save_strategy="steps",
    save_steps=4000,         
    save_total_limit=2,      
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    report_to="none",
    
    
    dataloader_num_workers=2,  
    dataloader_pin_memory=True,
    
    gradient_checkpointing=True, 
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("🚀 Bắt đầu training VI-EN OPTIMIZED...")
print(f"📁 Output: {config.output_dir}")
print(f"📊 LoRA r={config.lora_r}, alpha={config.lora_alpha}")
print(f"📦 Data: {len(train_dataset):,} samples (150k)")
print(f"⏱️ Epochs: {config.num_epochs}, Batch: {config.batch_size}")
print(f"⚡ Estimated time: ~8-10 hours (thay vì 82h)")
print(f"🎯 Target BLEU: 30-33\n")

# --- 8. AUTO-RESUME ---
def find_checkpoint():
    
    import json
    
    checkpoint = None
    final_model_path = os.path.join(config.output_dir, "final_model")
    
    
    if os.path.exists(final_model_path):
        print(f"✅ Model đã train xong tại {final_model_path}")
        return None, True  # (checkpoint, is_final)
    
  
    if os.path.exists(config.checkpoint_dataset_path):
        print(f"🔍 Tìm checkpoint trong dataset: {config.checkpoint_dataset_path}")
        
       
        dataset_checkpoints = []
        for root, dirs, files in os.walk(config.checkpoint_dataset_path):
            for item in dirs:
               
                if item.startswith("checkpoint-"):
                    item_path = os.path.join(root, item)
                    
                    adapter_config = os.path.join(item_path, "adapter_config.json")
                    trainer_state = os.path.join(item_path, "trainer_state.json")
                    
                    if os.path.exists(adapter_config) and os.path.exists(trainer_state):
                        dataset_checkpoints.append((item_path, item))
                        print(f"   Found valid checkpoint: {item}")
        
        if dataset_checkpoints:
            
            def get_checkpoint_step(item):
                path, name = item
                return int(name.split("-")[1])
            
            dataset_checkpoints.sort(key=get_checkpoint_step, reverse=True)
            checkpoint_path, checkpoint_name = dataset_checkpoints[0]
            
            
            config_file = os.path.join(checkpoint_path, "adapter_config.json")
            with open(config_file, 'r') as f:
                ckpt_config = json.load(f)
            
            ckpt_r = ckpt_config.get('r')
            ckpt_alpha = ckpt_config.get('lora_alpha')
            
            if ckpt_r == config.lora_r and ckpt_alpha == config.lora_alpha:
                print(f"✅ Compatible checkpoint found: {checkpoint_name}")
                print(f"   Config: r={ckpt_r}, alpha={ckpt_alpha}")
                checkpoint = checkpoint_path
            else:
                print(f"⚠️ Checkpoint config mismatch:")
                print(f"   Dataset: r={ckpt_r}, alpha={ckpt_alpha}")
                print(f"   Current: r={config.lora_r}, alpha={config.lora_alpha}")
                print(f"   → Starting fresh training")
    
    if checkpoint is None and os.path.exists(config.output_dir):
        print(f"🔍 Tìm checkpoint trong output directory: {config.output_dir}")
        
        checkpoints = []
        for d in os.listdir(config.output_dir):
            if d.startswith("checkpoint-"):
                checkpoint_path = os.path.join(config.output_dir, d)
                
                
                adapter_config = os.path.join(checkpoint_path, "adapter_config.json")
                trainer_state = os.path.join(checkpoint_path, "trainer_state.json")
                
                if os.path.exists(adapter_config) and os.path.exists(trainer_state):
                    checkpoints.append(d)
        
        if checkpoints:
            latest_checkpoint = max(checkpoints, key=lambda x: int(x.split("-")[1]))
            checkpoint_path = os.path.join(config.output_dir, latest_checkpoint)
            
            
            config_file = os.path.join(checkpoint_path, "adapter_config.json")
            with open(config_file, 'r') as f:
                ckpt_config = json.load(f)
            
            if ckpt_config.get('r') == config.lora_r and ckpt_config.get('lora_alpha') == config.lora_alpha:
                print(f"✅ Found local checkpoint: {latest_checkpoint}")
                checkpoint = checkpoint_path
            else:
                print(f"⚠️ Local checkpoint config mismatch, starting fresh")
    
    if checkpoint is None:
        print(f"▶️ No valid checkpoint found, starting fresh training")
        print(f"   Config: r={config.lora_r}, alpha={config.lora_alpha}")
        print(f"   📌 Note: Only checkpoint-XXXX (not final_model) can be resumed")
    
    return checkpoint, False
# Tìm checkpoint
checkpoint, is_final = find_checkpoint()

# Training
if not is_final:
    print("\n" + "="*60)
    if checkpoint:
        print(f"🔄 RESUMING TRAINING FROM CHECKPOINT")
        print(f"📁 Checkpoint: {checkpoint}")
        trainer.train(resume_from_checkpoint=checkpoint)
    else:
        print(f"🆕 STARTING FRESH TRAINING")
        trainer.train()
    print("="*60 + "\n")
    
    # Save final model
    final_path = os.path.join(config.output_dir, "final_model")
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"✅ Model saved to {final_path}")
    
    # ✅ SAVE BEST CHECKPOINT RIÊNG
    best_path = os.path.join(config.output_dir, "best_model")
    trainer.save_model(best_path)
    tokenizer.save_pretrained(best_path)
    print(f"✅ Best model saved to {best_path}")
else:
    print("⏭️ Training skipped (final_model exists)")

print("\n" + "="*60)
print("🎉 TRAINING COMPLETED!")
print("="*60)
print("📋 Next steps:")
print("1. Run BLEU evaluation với best_model")
print("2. So sánh BLEU score: Target > 35 (từ 27.47)")
print("3. Nếu chưa đạt, tăng model size lên 1.5B")
print("="*60)

✅ Hệ thống: 1 GPU


2025-12-23 06:22:38.776330: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766470958.797800     131 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766470958.804266     131 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766470958.821487     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766470958.821508     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766470958.821511     131 computation_placer.cc:177] computation placer alr

📂 Loading dữ liệu VI-EN...
📊 Sử dụng 150000 samples
✅ Train: 142500 | Val: 7500

=== VERIFY PROMPT ===
System: Translate Vietnamese medical text to English accurately and concisely.
User: Mục tiêu nghiên cứu: Đánh giá kết quả việc áp dụng can thiệp nội mạch trong điều trị rò động mạch cả...
Assistant: Objectives: To evaluate endovascular treatment for carotid cavernous fistalas....



tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 17,596,416 || all params: 511,629,184 || trainable%: 3.4393
🚀 Bắt đầu training VI-EN OPTIMIZED...
📁 Output: /kaggle/working/qwen_mt_vi_en_v2
📊 LoRA r=32, alpha=64
📦 Data: 142,500 samples (150k)
⏱️ Epochs: 3, Batch: 8
⚡ Estimated time: ~8-10 hours (thay vì 82h)
🎯 Target BLEU: 30-33

🔍 Tìm checkpoint trong dataset: /kaggle/input/checkpoint/
   Found valid checkpoint: checkpoint-12000
   Found valid checkpoint: checkpoint-13362
✅ Compatible checkpoint found: checkpoint-13362
   Config: r=32, alpha=64

🔄 RESUMING TRAINING FROM CHECKPOINT
📁 Checkpoint: /kaggle/input/checkpoint/qwen_mt_vi_en_v2/checkpoint-13362


Could not locate the best model at /kaggle/working/qwen_mt_vi_en_v2/checkpoint-12000/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


Step,Training Loss,Validation Loss



✅ Model saved to /kaggle/working/qwen_mt_vi_en_v2/final_model
✅ Best model saved to /kaggle/working/qwen_mt_vi_en_v2/best_model

🎉 TRAINING COMPLETED!
📋 Next steps:
1. Run BLEU evaluation với best_model
2. So sánh BLEU score: Target > 35 (từ 27.47)
3. Nếu chưa đạt, tăng model size lên 1.5B


In [3]:
!pip install evaluate sacrebleu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import Dataset
from tqdm import tqdm
import evaluate
import re
import warnings

# Tắt warnings
warnings.filterwarnings('ignore')

# --- CONFIG ---
class config:
    data_dir = "/kaggle/input/vlsp-data"
    base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
   
    checkpoint_path = "/kaggle/working/qwen_mt_vi_en_v2/final_model"
    
   
    use_public_test = True
    test_src_file = "public_test.vi" if use_public_test else "test.vi"
    test_ref_file = "public_test.en" if use_public_test else "test.en"
    
    max_samples = None  

# --- 1. LOAD DỮ LIỆU ---
def load_test_data(src_filename, trg_filename):
    """Load dữ liệu test từ file"""
    src_path, trg_path = "", ""
    
    
    src_variants = [src_filename, f"{src_filename}.txt"]
    trg_variants = [trg_filename, f"{trg_filename}.txt"]
    
    for root, dirs, files in os.walk(config.data_dir):
        for src_var in src_variants:
            if src_var in files and not src_path:
                src_path = os.path.join(root, src_var)
        for trg_var in trg_variants:
            if trg_var in files and not trg_path:
                trg_path = os.path.join(root, trg_var)
    
    if not src_path or not trg_path:
        print(f"\n🔍 Files found in {config.data_dir}:")
        for root, dirs, files in os.walk(config.data_dir):
            for f in files:
                if 'test' in f.lower():
                    print(f"   - {f}")
        raise FileNotFoundError(f"Không tìm thấy {src_filename} hoặc {trg_filename}")
    
    with open(src_path, 'r', encoding='utf-8') as f:
        src_lines = [l.strip() for l in f if l.strip()]
    with open(trg_path, 'r', encoding='utf-8') as f:
        trg_lines = [l.strip() for l in f if l.strip()]
    
    return list(zip(src_lines, trg_lines))

def clean_text(text):
    """Làm sạch text"""
    text = ' '.join(text.split())
    return re.sub(r'\s+([.,!?;:])', r'\1', text).strip()

# --- 2. LOAD MODEL ---
print("="*60)
print("🔧 LOADING MODEL")
print("="*60)
print(f"Checkpoint: {config.checkpoint_path}")

# Kiểm tra loại checkpoint
is_merged_model = os.path.exists(os.path.join(config.checkpoint_path, "config.json"))
is_adapter = os.path.exists(os.path.join(config.checkpoint_path, "adapter_config.json"))

if not is_merged_model and not is_adapter:
    raise ValueError(f"Invalid checkpoint path: {config.checkpoint_path}")

print(f"Type: {'Merged Model' if is_merged_model else 'LoRA Adapter'}\n")

print("🔄 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    config.base_model_name,
    trust_remote_code=True,
    padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if is_merged_model:
    # Load merged model trực tiếp
    print("🔄 Loading merged model...")
    model = AutoModelForCausalLM.from_pretrained(
        config.checkpoint_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
else:
    # Load base model + adapter
    print("🔄 Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    
    print("🔄 Loading LoRA adapter from checkpoint...")
    model = PeftModel.from_pretrained(base_model, config.checkpoint_path)
    
    

model.eval()
print("✅ Model loaded successfully!\n")

# --- 3. HÀM TRANSLATE ---
def translate_batch(sources, batch_size=8):
    """Translate batch Vietnamese → English"""
    translations = []
    
    for i in tqdm(range(0, len(sources), batch_size), desc="Translating", ncols=80):
        batch = sources[i:i+batch_size]
        
        
        messages_batch = []
        for src in batch:
            messages = [
                {
                    "role": "system",
                    "content": "Translate Vietnamese medical text to English accurately and concisely."
                },
                {
                    "role": "user",
                    "content": src
                }
            ]
            messages_batch.append(messages)
        
        # Apply chat template
        prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages_batch
        ]
        
        # Tokenize
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                num_beams=4,
                length_penalty=0.9,
                early_stopping=True,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Decode
        for j, output in enumerate(outputs):
            input_length = inputs.input_ids[j].shape[0]
            generated = output[input_length:]
            translation = tokenizer.decode(generated, skip_special_tokens=True)
            translations.append(translation.strip())
    
    return translations

# --- 4. TÍNH BLEU ---
print("="*60)
print("📊 LOADING TEST DATA")
print("="*60)
print(f"Dataset: {'PUBLIC TEST' if config.use_public_test else 'PRIVATE TEST'}")
print(f"Source: {config.test_src_file}")
print(f"Reference: {config.test_ref_file}")

try:
    test_data = load_test_data(config.test_src_file, config.test_ref_file)
except FileNotFoundError as e:
    print(f"\n❌ Error: {e}")
    raise

if config.max_samples:
    test_data = test_data[:config.max_samples]
    print(f"Samples: {len(test_data)} (limited)")
else:
    print(f"Samples: {len(test_data)} (all)")
print("="*60 + "\n")

sources = [clean_text(s) for s, _ in test_data]
references = [clean_text(t) for _, t in test_data]

print("🚀 Starting translation...")
predictions = translate_batch(sources, batch_size=8)

# Load BLEU
print("\n📈 Calculating BLEU score...")
bleu = evaluate.load("bleu")

# Format references
references_formatted = [[ref] for ref in references]

results = bleu.compute(
    predictions=predictions,
    references=references_formatted,
    max_order=4
)

# --- 5. HIỂN THỊ KẾT QUẢ ---
print("\n" + "="*60)
print("📊 BLEU SCORE RESULTS")
print("="*60)
print(f"Checkpoint: {config.checkpoint_path}")
print(f"Dataset: {config.test_src_file}")
print("-"*60)
print(f"BLEU-4: {results['bleu']*100:.2f}")
print(f"BLEU-1: {results['precisions'][0]*100:.2f}")
print(f"BLEU-2: {results['precisions'][1]*100:.2f}")
print(f"BLEU-3: {results['precisions'][2]*100:.2f}")
print(f"Brevity Penalty: {results['brevity_penalty']:.4f}")
print(f"Length Ratio: {results['length_ratio']:.4f}")
print("="*60)

# --- 6. SAMPLE TRANSLATIONS (✅ FIXED: 5 samples, full text) ---
print("\n📝 SAMPLE TRANSLATIONS (first 5):")
print("="*60)
for i in range(min(5, len(sources))):
    print(f"\n--- Example {i+1} ---")
    print(f"VI:   {sources[i]}")
    print(f"REF:  {references[i]}")
    print(f"PRED: {predictions[i]}")
    print("-"*60)

# --- 7. LƯU KẾT QUẢ ---
checkpoint_name = os.path.basename(config.checkpoint_path)
dataset_type = "public_test" if config.use_public_test else "private_test"
output_file = f"/kaggle/working/bleu_{checkpoint_name}_{dataset_type}.txt"

with open(output_file, 'w', encoding='utf-8') as f:
    f.write(f"BLEU SCORE RESULTS - {checkpoint_name}\n")
    f.write("="*60 + "\n")
    f.write(f"Checkpoint: {config.checkpoint_path}\n")
    f.write(f"Dataset: {config.test_src_file} -> {config.test_ref_file}\n")
    f.write(f"Number of samples: {len(sources)}\n")
    f.write("="*60 + "\n\n")
    f.write(f"BLEU-4: {results['bleu']*100:.2f}\n")
    f.write(f"BLEU-1: {results['precisions'][0]*100:.2f}\n")
    f.write(f"BLEU-2: {results['precisions'][1]*100:.2f}\n")
    f.write(f"BLEU-3: {results['precisions'][2]*100:.2f}\n")
    f.write(f"Brevity Penalty: {results['brevity_penalty']:.4f}\n")
    f.write(f"Length Ratio: {results['length_ratio']:.4f}\n")
    f.write("="*60 + "\n\n")
    
    f.write("SAMPLE TRANSLATIONS\n")
    f.write("="*60 + "\n")
    for i in range(min(10, len(sources))):
        f.write(f"\n--- Example {i+1} ---\n")
        f.write(f"Source: {sources[i]}\n")
        f.write(f"Reference: {references[i]}\n")
        f.write(f"Predicted: {predictions[i]}\n")
        f.write("-"*60 + "\n")

print(f"\n✅ Results saved to {output_file}")

# --- 8. SACREBLEU ---
try:
    import sacrebleu
    
    refs_sacre = [[ref] for ref in references]
    refs_transposed = list(zip(*refs_sacre))
    
    sacre_score = sacrebleu.corpus_bleu(predictions, refs_transposed)
    
    print("\n" + "="*60)
    print("📊 SACREBLEU SCORE (Standard)")
    print("="*60)
    print(f"SacreBLEU: {sacre_score.score:.2f}")
    print(f"Signature: {sacre_score.format()}")
    print("="*60)
    
except ImportError:
    print("\n⚠️ SacreBLEU not installed: pip install sacrebleu")

# --- 9. SO SÁNH VỚI BASELINE ---
print("\n" + "="*60)
print("📊 COMPARISON WITH BASELINE")
print("="*60)
print(f"Baseline (checkpoint-4000): BLEU = 27.47")
print(f"Current  ({checkpoint_name}):  BLEU = {results['bleu']*100:.2f}")
improvement = results['bleu']*100 - 27.47
print(f"Improvement:   {improvement:+.2f} points")
if results['bleu']*100 >= 35:
    print("✅ TARGET ACHIEVED! (BLEU ≥ 35)")
elif results['bleu']*100 >= 30:
    print("⚠️  Good progress, close to target")
else:
    print("📈 Continue training or try larger model")
print("="*60)

🔧 LOADING MODEL
Checkpoint: /kaggle/working/qwen_mt_vi_en_v2/final_model
Type: LoRA Adapter

🔄 Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


🔄 Loading base model...
🔄 Loading LoRA adapter from checkpoint...
✅ Model loaded successfully!

📊 LOADING TEST DATA
Dataset: PUBLIC TEST
Source: public_test.vi
Reference: public_test.en
Samples: 3000 (all)

🚀 Starting translation...


Translating: 100%|████████████████████████████| 375/375 [32:14<00:00,  5.16s/it]



📈 Calculating BLEU score...



📊 BLEU SCORE RESULTS
Checkpoint: /kaggle/working/qwen_mt_vi_en_v2/final_model
Dataset: public_test.vi
------------------------------------------------------------
BLEU-4: 33.05
BLEU-1: 63.05
BLEU-2: 38.65
BLEU-3: 26.56
Brevity Penalty: 0.9908
Length Ratio: 0.9909

📝 SAMPLE TRANSLATIONS (first 5):

--- Example 1 ---
VI:   Thực trạng kiến thức và thực hành của người có thẻ bảo hiểm y tế trong sử dụng dịch vụ khám chữa bệnh ở các cơ sở y tế công và một số yếu tố ảnh hưởng tại tỉnh Viêng Chăn, CHDCND Lào, năm 2017
REF:  Knowledge, practices in public health service utilization among health insurance card’s holders and influencing factors in Vientiane, Lao
PRED: The status of knowledge and practice of people with health insurance cards in using medical examination and treatment services in public and some factors affecting in Vien Chan province, Lao Cai province, 2017
------------------------------------------------------------

--- Example 2 ---
VI:   Mô tả thực trạng kiến thức, thực hành

In [6]:
import evaluate
import sacrebleu

print("="*60)
print("📊 MT EVALUATION METRICS")
print("="*60)

# ---------- BLEU (evaluate) ----------
bleu = evaluate.load("bleu")
refs_bleu = [[r] for r in references]

bleu_res = bleu.compute(
    predictions=predictions,
    references=refs_bleu
)

print(f"BLEU-4 (evaluate): {bleu_res['bleu']*100:.2f}")
print(f"BLEU-1: {bleu_res['precisions'][0]*100:.2f}")
print(f"BLEU-2: {bleu_res['precisions'][1]*100:.2f}")
print(f"BLEU-3: {bleu_res['precisions'][2]*100:.2f}")
print(f"BP: {bleu_res['brevity_penalty']:.4f}")
print("-"*60)

# ---------- SacreBLEU (standard) ----------
refs_sacre = [references]  # list[list[str]]
sacre_bleu = sacrebleu.corpus_bleu(predictions, refs_sacre)

print(f"SacreBLEU: {sacre_bleu.score:.2f}")
print(f"Signature: {sacre_bleu.format()}")
print("-"*60)

# ---------- TER ----------
ter = sacrebleu.corpus_ter(predictions, refs_sacre)
print(f"TER (lower is better): {ter.score:.2f}")
print("-"*60)

# ---------- METEOR ----------
meteor = evaluate.load("meteor")
meteor_res = meteor.compute(
    predictions=predictions,
    references=references
)
print(f"METEOR: {meteor_res['meteor']*100:.2f}")
print("-"*60)

# ---------- chrF++ ----------
chrf = sacrebleu.corpus_chrf(
    predictions,
    refs_sacre,
    word_order=2  # chrF++
)
print(f"chrF++: {chrf.score:.2f}")

print("="*60)


📊 MT EVALUATION METRICS
BLEU-4 (evaluate): 33.05
BLEU-1: 63.05
BLEU-2: 38.65
BLEU-3: 26.56
BP: 0.9908
------------------------------------------------------------
SacreBLEU: 33.05
Signature: BLEU = 33.05 63.1/38.6/26.6/19.1 (BP = 0.991 ratio = 0.991 hyp_len = 75903 ref_len = 76602)
------------------------------------------------------------
TER (lower is better): 61.29
------------------------------------------------------------


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR: 61.59
------------------------------------------------------------
chrF++: 55.32
